# Парсинг статей Habr из PDF
Извлекаем: название, компанию, рейтинг, автора, теги, хабы, текст статьи

In [2]:
# Установка зависимостей (если нет)
# !pip install PyMuPDF pandas

In [3]:
import fitz  # PyMuPDF
import pandas as pd
import glob
import re
import os

In [4]:
# ============================
# НАСТРОЙКИ — поменяй путь
# ============================
PDF_DIR = os.path.expanduser("~/Projects/MOiBD/pdf_articles/")
OUTPUT_CSV = os.path.expanduser("~/Projects/MOiBD/habr_articles.csv")

all_pdf = glob.glob(PDF_DIR + "*.pdf")
print(f"Найдено PDF: {len(all_pdf)}")

Найдено PDF: 102


In [5]:
def extract_pages(pdf_path):
    """Извлечь текст всех страниц"""
    doc = fitz.open(pdf_path)
    pages = [doc.load_page(i).get_text("text") for i in range(len(doc))]
    doc.close()
    return pages

In [6]:
def parse_habr_pdf(pages, filename):
    """
    Реальная структура страницы 0 у Habr PDF (два варианта):

    Вариант A — статья от компании (есть блок компании сверху):
      line 0: НазваниеКомпании
      line 1: Описание компании
      line 2: '64,97 Рейтинг'
      line 3: '218 Подписчики'
      line 4: 'Подписаться'
      line 5..N: текст начала статьи
      line N+1: username_автора
      line N+2: 'X минут назад' / дата
      line N+3: ЗАГОЛОВОК (может быть многострочным)
      line N+4: 'Простой'/'Средний'/'Сложный'

    Вариант B — личный блог (нет блока компании):
      line 0: username_автора
      line 1: 'X минут назад' / дата
      line 2: ЗАГОЛОВОК
      line 3: 'Простой'/'Средний'/'Сложный'
    """
    result = {
        "filename": os.path.basename(filename),
        "title": "",
        "company": "",
        "company_description": "",
        "rating": "",
        "subscribers": "",
        "author": "",
        "publish_time": "",
        "read_time": "",
        "views": "",
        "hubs": "",
        "tags": "",
        "text": ""
    }

    if not pages:
        return result

    page0 = pages[0]
    lines = [l.strip() for l in page0.split("\n") if l.strip()]
    full_text = "\n".join(pages)

    # ======================
    # 1. Компания и рейтинг
    # Паттерн: 'XX,XX Рейтинг' — всегда в одной строке
    # ======================
    rating_idx = None
    for i, line in enumerate(lines):
        m = re.match(r'^([\d,\.]+)\s+Рейтинг$', line)
        if m:
            result["rating"] = m.group(1)
            rating_idx = i
            # компания — две строки выше рейтинга
            if i >= 2:
                result["company"] = lines[i - 2]
                result["company_description"] = lines[i - 1]
            elif i == 1:
                result["company"] = lines[0]
            break

    # ======================
    # 2. Подписчики
    # ======================
    for line in lines:
        m = re.match(r'^([\d\s]+)\s+Подписчик', line)
        if m:
            result["subscribers"] = m.group(1).strip()
            break

    # ======================
    # 3. Автор и время
    # Ищем: строка = только латиница/цифры/подчёркивание (логин)
    # + следующая содержит 'назад' или дату
    # ======================
    time_pattern = re.compile(
        r'(назад|\d+\s*(мин|час|дн|сек)|'
        r'\d+\s+(янв|фев|мар|апр|май|июн|июл|авг|сен|окт|ноя|дек))'
    )
    for i, line in enumerate(lines):
        if re.match(r'^[a-zA-Z0-9_]+$', line) and i + 1 < len(lines):
            if time_pattern.search(lines[i + 1]):
                result["author"] = line
                result["publish_time"] = lines[i + 1]
                # заголовок идёт после времени
                author_idx = i
                break

    # ======================
    # 4. Заголовок статьи
    # Идёт после строки с временем публикации
    # Может быть многострочным — собираем до строки с метаданными
    # ======================
    stop_words = {"Простой", "Средний", "Сложный", "Подписаться", "Комментировать"}
    stop_pattern = re.compile(r'^(\d+\s*мин|\d+\s*K|\d{3,}|Блог компании|Хабы:|Теги:)')

    if result["publish_time"]:
        pub_idx = None
        for i, line in enumerate(lines):
            if line == result["publish_time"]:
                pub_idx = i
                break
        
        if pub_idx is not None:
            title_parts = []
            for line in lines[pub_idx + 1:]:
                if line in stop_words or stop_pattern.match(line):
                    break
                if len(line) > 3:
                    title_parts.append(line)
                if len(title_parts) >= 3:  # заголовок не длиннее 3 строк
                    break
            result["title"] = " ".join(title_parts).strip()

    # Fallback: заголовок из имени файла
    if not result["title"]:
        name = os.path.basename(filename)
        name = re.sub(r'^\d+_', '', name)
        name = re.sub(r'\s*_?\s*Хабр\.pdf$', '', name, flags=re.IGNORECASE)
        name = name.replace(".pdf", "")
        result["title"] = name.strip()

    # ======================
    # 5. Время чтения и просмотры
    # Паттерн: 'X мин' и число просмотров рядом с заголовком
    # ======================
    for line in lines:
        m = re.match(r'^(\d+)\s+мин$', line)
        if m:
            result["read_time"] = m.group(1)
            break

    # ======================
    # 6. Хабы и теги — ищем по всему тексту
    # ======================
    # Хабы: строка вида 'Блог компании iSpring,\xa0Тестирование IT-систем*,...'
    for line in lines:
        if re.search(r'Блог компании|Тестирование|Программирование|Разработка', line):
            # чистим \xa0 и *
            cleaned = line.replace('\xa0', ' ').replace('*', '').strip()
            if len(cleaned) > 5:
                result["hubs"] = cleaned
                break

    tags_match = re.search(r'Теги:\s*(.+?)(?:\n|Хабы:|$)', full_text, re.DOTALL)
    if tags_match:
        tags_raw = tags_match.group(1).replace('\n', ' ').strip()
        result["tags"] = re.sub(r'\s+', ' ', tags_raw)

    hubs_match = re.search(r'Хабы:\s*(.+?)(?:\n\n|$)', full_text, re.DOTALL)
    if hubs_match and not result["hubs"]:
        result["hubs"] = hubs_match.group(1).replace('\n', ' ').strip()

    # ======================
    # 7. Полный текст
    # ======================
    result["text"] = full_text.replace("\n", " ").strip()

    return result

In [7]:
# ============================
# Проверка на одном файле
# ============================
test_pdf = all_pdf[0]
print("Файл:", os.path.basename(test_pdf))
print()

pages = extract_pages(test_pdf)
parsed = parse_habr_pdf(pages, test_pdf)

for k, v in parsed.items():
    if k != "text":
        print(f"{k:25s}: {v}")
print(f"{'text':25s}: {parsed['text'][:200]}...")

Файл: 75_ИИ найди факты а я подумаю почему гибридный подход не работает для форсайта  Хабр.pdf

filename                 : 75_ИИ найди факты а я подумаю почему гибридный подход не работает для форсайта  Хабр.pdf
title                    : «ИИ, найди факты, а я подумаю»: почему гибридный подход не работает для форсайта
company                  : 
company_description      : 
rating                   : 
subscribers              : 
author                   : rowtheboat8
publish_time             : 18 часов назад
read_time                : 6
views                    : 
hubs                     : Искусственный интеллект, Будущее здесь Электропочта Подписаться 0 3 0
tags                     : искусственный интеллект, форсайт, аналитика данных
text                     : В прошлом году у меня вышла небольшая заметка с рассуждениями на тему потенциала использования аналитических систем на базе ИИ для задач прогнозирования и форсайта с очень сдержанными оценками его реа...


In [8]:
# ============================
# Отладка: первые 30 строк page0 любого PDF
# ============================
# Меняй индекс для проверки разных файлов
debug_idx = 5  # <-- меняй

debug_pdf = all_pdf[debug_idx]
print("Файл:", os.path.basename(debug_pdf))
doc = fitz.open(debug_pdf)
lines = [l.strip() for l in doc.load_page(0).get_text("text").split("\n") if l.strip()]
doc.close()
for i, line in enumerate(lines[:30]):
    print(f"{i:3d}: {repr(line)}")

Файл: 18_Шифруем ID сетью Фейстеля защита API без правок в базе  Хабр.pdf
  0: 'Рабочий демо-проект: github.com/mlivirov/fiestel-cipher-demo'
  1: 'Я работаю контрактором и регулярно попадаю в чужие кодовые базы. И очень во многих из них'
  2: '— по вполне понятным причинам: денег нет, людей нет, сроки вчера, код достался от кого-то'
  3: 'ещё — выбор почти всегда в пользу «проще», а не «безопаснее». Статья написана именно для'
  4: 'таких проектов. Это не агитация «делайте безопасность по-человечески» (надо, конечно). Это'
  5: 'конкретный приём, который делается за вечер, в рантайме стоит буквально ничего и закрывает'
  6: 'целый класс тривиальных атак, которые до сих пор массово встречаются в продакшене.'
  7: 'Куча веб-приложений светит в URL целочисленные первичные ключи:'
  8: 'GET /invoices/1043'
  9: 'GET /invoices/1044'
 10: 'GET /invoices/1045'
 11: 'Один валидный URL плюс цикл for  — и у тебя на руках вся таблица. Ровно на этом в апреле'
 12: '2025 года погорела APCOA, утёкш

In [9]:
# ============================
# Парсим все PDF
# ============================
records = []
errors = []

for pdf_path in all_pdf:
    try:
        pages = extract_pages(pdf_path)
        record = parse_habr_pdf(pages, pdf_path)
        records.append(record)
        status = f"R={record['rating'] or '—':6s} C={record['company'][:20] if record['company'] else '—'}"
        print(f"✓ {status} | {os.path.basename(pdf_path)[:50]}")
    except Exception as e:
        errors.append({"file": pdf_path, "error": str(e)})
        print(f"✗ {os.path.basename(pdf_path)[:60]} → {e}")

print(f"\nОбработано: {len(records)}, ошибок: {len(errors)}")

✓ R=—      C=— | 75_ИИ найди факты а я подумаю почему гибридный под
✓ R=1      C=BetBoom | 74_Агенты везде Разработчики в опасности  Хабр.pdf
✓ R=128,28 C=«Лаборатория Касперс | 41_Security Week 2617 криптостилеры в китайском Ap
✓ R=—      C=— | 38_Выделение регистров процессора при помощи генет
✓ R=—      C=— | 28_Современный дата-стек потоковая система из LEGO
✓ R=—      C=— | 18_Шифруем ID сетью Фейстеля защита API без правок
✓ R=—      C=— | 54_От PLC к своему HMI и AI-анализу  Хабр.pdf
✓ R=—      C=— | 73_Как снять дамп на маршрутизаторах Huawei за 1 м
✓ R=—      C=— | 96_40 команд от Яндекс Smart Home за 3 секунды как
✓ R=—      C=— | 30_Молодость всё прощает  Хабр.pdf
✓ R=279,65 C=SimpleOne | 94_Как не потерять тысячу ноутбуков  Хабр.pdf
✓ R=—      C=— | 100_Патент на промышленный образец или авторское п
✓ R=—      C=— | 60_Грядущий Великий Переход 20  Хабр.pdf
✓ R=—      C=— | 55_За пределами неба История основателя Hyperliqui
✓ R=279,65 C=SimpleOne | 51_Почему ваш бэклог давно

In [10]:
# Создаём DataFrame и смотрим статистику
df = pd.DataFrame(records)

total = len(df)
has_company = df["company"].notna() & (df["company"] != "")
has_rating  = df["rating"].notna()  & (df["rating"]  != "")
has_author  = df["author"].notna()  & (df["author"]  != "")

print(f"Всего статей  : {total}")
print(f"С компанией   : {has_company.sum()} ({100*has_company.sum()//total}%)")
print(f"С рейтингом   : {has_rating.sum()}  ({100*has_rating.sum()//total}%)")
print(f"С автором     : {has_author.sum()}  ({100*has_author.sum()//total}%)")
print()
df[["filename", "title", "company", "rating", "author"]].head(15)

Всего статей  : 102
С компанией   : 38 (37%)
С рейтингом   : 38  (37%)
С автором     : 98  (96%)



,filename,title,company,rating,author
0,75_ИИ найди факты а я подумаю почему гибридный...,"«ИИ, найди факты, а я подумаю»: почему гибридн...",,,rowtheboat8
1,74_Агенты везде Разработчики в опасности Хабр...,Агенты везде. Разработчики в опасности…,BetBoom,1,itatarchenkoru
2,41_Security Week 2617 криптостилеры в китайско...,Security Week 2617: криптостилеры в китайском ...,«Лаборатория Касперского»,"128,28",Kaspersky_Lab
3,38_Выделение регистров процессора при помощи г...,Выделение регистров процессора при помощи гене...,,,Sivchenko_translate
4,28_Современный дата-стек потоковая система из ...,Современный дата-стек: потоковая система из «L...,,,mirongaskov
5,18_Шифруем ID сетью Фейстеля защита API без пр...,Шифруем ID сетью Фейстеля: защита API без прав...,,,livirov
6,54_От PLC к своему HMI и AI-анализу Хабр.pdf,От PLC к своему HMI и AI-анализу,,,vladipirogov
7,73_Как снять дамп на маршрутизаторах Huawei за...,Как снять дамп на маршрутизаторах Huawei за 1 ...,,,Denis_Vdovin
8,96_40 команд от Яндекс Smart Home за 3 секунды...,40 команд от Яндекс Smart Home за 3 секунды: к...,,,mr_MAIL
9,30_Молодость всё прощает Хабр.pdf,Молодость всё прощает?,,,zingilevskiy


In [11]:
# Статьи БЕЗ компании (личные блоги — это нормально)
df[~has_company][["filename", "title", "author", "rating"]].head(20)

,filename,title,author,rating
0,75_ИИ найди факты а я подумаю почему гибридный...,"«ИИ, найди факты, а я подумаю»: почему гибридн...",rowtheboat8,
3,38_Выделение регистров процессора при помощи г...,Выделение регистров процессора при помощи гене...,Sivchenko_translate,
4,28_Современный дата-стек потоковая система из ...,Современный дата-стек: потоковая система из «L...,mirongaskov,
5,18_Шифруем ID сетью Фейстеля защита API без пр...,Шифруем ID сетью Фейстеля: защита API без прав...,livirov,
6,54_От PLC к своему HMI и AI-анализу Хабр.pdf,От PLC к своему HMI и AI-анализу,vladipirogov,
7,73_Как снять дамп на маршрутизаторах Huawei за...,Как снять дамп на маршрутизаторах Huawei за 1 ...,Denis_Vdovin,
8,96_40 команд от Яндекс Smart Home за 3 секунды...,40 команд от Яндекс Smart Home за 3 секунды: к...,mr_MAIL,
9,30_Молодость всё прощает Хабр.pdf,Молодость всё прощает?,zingilevskiy,
11,100_Патент на промышленный образец или авторск...,Патент на промышленный образец или авторское п...,lireate,
12,60_Грядущий Великий Переход 20 Хабр.pdf,Грядущий Великий Переход 2.0,TimurSadekov,


In [12]:
# Сохраняем
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Сохранено: {OUTPUT_CSV}")
print(f"Строк: {len(df)}, колонок: {len(df.columns)}")
print(f"Колонки: {list(df.columns)}")

Сохранено: /home/yaekitsune/Projects/MOiBD/habr_articles.csv
Строк: 102, колонок: 13
Колонки: ['filename', 'title', 'company', 'company_description', 'rating', 'subscribers', 'author', 'publish_time', 'read_time', 'views', 'hubs', 'tags', 'text']


In [13]:
# Проверка
df2 = pd.read_csv(OUTPUT_CSV)
df2[["filename", "title", "company", "rating", "tags"]].head(10)

,filename,title,company,rating,tags
0,75_ИИ найди факты а я подумаю почему гибридный...,"«ИИ, найди факты, а я подумаю»: почему гибридн...",NaN,NaN,"искусственный интеллект, форсайт, аналитика да..."
1,74_Агенты везде Разработчики в опасности Хабр...,Агенты везде. Разработчики в опасности…,BetBoom,1,"конференция, ии-агенты, вайб-кодинг, vendor lo..."
2,41_Security Week 2617 криптостилеры в китайско...,Security Week 2617: криптостилеры в китайском ...,«Лаборатория Касперского»,"128,28","apple, иб, app store, ledger"
3,38_Выделение регистров процессора при помощи г...,Выделение регистров процессора при помощи гене...,NaN,NaN,"исследования, оптимизация, процессор, алгоритм..."
4,28_Современный дата-стек потоковая система из ...,Современный дата-стек: потоковая система из «L...,NaN,NaN,"Kafka, Flink, Redis, Iceberg, Java, Docker, do..."
5,18_Шифруем ID сетью Фейстеля защита API без пр...,Шифруем ID сетью Фейстеля: защита API без прав...,NaN,NaN,"шифрование, сеть фейстеля, web, restapi"
6,54_От PLC к своему HMI и AI-анализу Хабр.pdf,От PLC к своему HMI и AI-анализу,NaN,NaN,"diy-проекты, orangepi, raspberrypi, llm-модели..."
7,73_Как снять дамп на маршрутизаторах Huawei за...,Как снять дамп на маршрутизаторах Huawei за 1 ...,NaN,NaN,"wireshark, анализатор трафика, huawei"
8,96_40 команд от Яндекс Smart Home за 3 секунды...,40 команд от Яндекс Smart Home за 3 секунды: к...,NaN,NaN,"алиса, homeassistant"
9,30_Молодость всё прощает Хабр.pdf,Молодость всё прощает?,NaN,NaN,"здоровье и образ жизни, вредные привычки, рути..."
